# 01 · Dataset EDA (UNFAIR-ToS)

Statistics and figures for the report. Every item of the EDA checklist in the implementation guide (section 6) has a cell below. Figures are saved to `My Drive/fineguard/figures/`.

## Setup
Clones the repo, installs packages, and links `models/` and `mlruns/` to Google Drive so nothing is lost if Colab disconnects.

In [ ]:
# 1. Get the code (for a private repo, see the README: add GITHUB_TOKEN in Colab Secrets)
import os
if not os.path.exists("/content/fineguard"):
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        !git clone -q https://{token}@github.com/424103-achu/fineguard.git /content/fineguard
    except Exception:
        !git clone -q https://github.com/424103-achu/fineguard.git /content/fineguard
%cd /content/fineguard/ml
!git pull -q
!pip install -q -r requirements.txt

In [ ]:
# 2. Save outputs to Google Drive so they survive a disconnect
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/fineguard"
!mkdir -p {DRIVE}/models {DRIVE}/mlruns {DRIVE}/figures
!rm -rf models mlruns && ln -sfn {DRIVE}/models models && ln -sfn {DRIVE}/mlruns mlruns

# MLflow's SQLite file works best on local disk: copy it from Drive now, back at the end
import os, shutil
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////content/mlflow.db"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
if os.path.exists(f"{DRIVE}/mlruns/mlflow.db"):
    shutil.copy(f"{DRIVE}/mlruns/mlflow.db", "/content/mlflow.db")
FIG = f"{DRIVE}/figures"

## Load the data
Label names come from the dataset features, never hardcoded. The chronological split is kept as released.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data import load_unfair_tos, get_split

ds, LABELS = load_unfair_tos()
N = len(LABELS)
print(ds)
print("Label order:", list(enumerate(LABELS)))
splits = {s: get_split(ds, s, N) for s in ["train", "validation", "test"]}

## Sentences per split and share of potentially unfair sentences

In [ ]:
rows = []
for s, (texts, y) in splits.items():
    rows.append({"split": s, "sentences": len(texts),
                 "potentially unfair": int(y.any(axis=1).sum()),
                 "% unfair": round(100 * y.any(axis=1).mean(), 2)})
split_table = pd.DataFrame(rows)
split_table

## Count per category in each split

In [ ]:
counts = pd.DataFrame({s: y.sum(axis=0) for s, (_, y) in splits.items()}, index=LABELS)
display(counts)
ax = counts.plot.barh(figsize=(8, 5))
ax.set_xlabel("sentences"); ax.set_title("Sentences per category and split")
plt.tight_layout(); plt.savefig(f"{FIG}/category_counts.png", dpi=150); plt.show()

## Number of labels per sentence

In [ ]:
pd.DataFrame({s: pd.Series(y.sum(axis=1)).value_counts().sort_index()
              for s, (_, y) in splits.items()}).fillna(0).astype(int).rename_axis("labels per sentence")

## Category co-occurrence (train)

In [ ]:
from src.error_analysis import co_occurrence
co = co_occurrence(splits["train"][1], LABELS)
plt.figure(figsize=(7, 6)); sns.heatmap(co, annot=True, fmt="d", cmap="Blues")
plt.title("Category co-occurrence (train)"); plt.tight_layout()
plt.savefig(f"{FIG}/cooccurrence.png", dpi=150); plt.show()

## Sentence length in words and in BERT tokens
The 95th/99th percentile of the token length decides `--max-len` for the transformers.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
texts_tr = splits["train"][0]
words = pd.Series([len(t.split()) for t in texts_tr])
tokens = pd.Series([len(ids) for ids in tok(texts_tr)["input_ids"]])
display(pd.DataFrame({"words": words.describe(percentiles=[.5, .95, .99]),
                      "legal-bert tokens": tokens.describe(percentiles=[.5, .95, .99])}))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
words.clip(upper=150).hist(bins=50, ax=axes[0]); axes[0].set_title("words per sentence (clipped at 150)")
tokens.clip(upper=200).hist(bins=50, ax=axes[1]); axes[1].set_title("Legal-BERT tokens (clipped at 200)")
plt.tight_layout(); plt.savefig(f"{FIG}/lengths.png", dpi=150); plt.show()
print("share of sentences longer than 128 tokens:", round((tokens > 128).mean() * 100, 2), "%")

## Top TF-IDF terms per category

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
texts, y = splits["train"]
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, stop_words="english", sublinear_tf=True)
X = vec.fit_transform(texts)
vocab = np.array(vec.get_feature_names_out())
for j, name in enumerate(LABELS):
    pos = np.asarray(X[y[:, j] == 1].mean(axis=0)).ravel()
    rest = np.asarray(X[y[:, j] == 0].mean(axis=0)).ravel()
    print(f"{name:25s}", ", ".join(vocab[np.argsort(pos - rest)[::-1][:12]]))

## Example sentences per category
Spot-check that the label order is what you think it is.

In [ ]:
rng = np.random.default_rng(0)
for j, name in enumerate(LABELS):
    idx = np.where(y[:, j] == 1)[0]
    print(f"\n=== {name} ({len(idx)} train sentences)")
    for i in rng.choice(idx, size=min(3, len(idx)), replace=False):
        print(" -", texts[i][:250])

## Is the text lowercased?

In [ ]:
all_texts = [t for s in splits.values() for t in s[0]]
has_upper = sum(any(c.isupper() for c in t) for t in all_texts)
print(f"sentences containing an uppercase letter: {has_upper} of {len(all_texts)}")

In [ ]:
split_table.to_csv(f"{FIG}/split_table.csv", index=False)
counts.to_csv(f"{FIG}/category_counts.csv")